In [ ]:
#@title Cell 0 — Setup (run me first)
# ============================================================
# Shared Infrastructure: company_sim.py (embedded)
# ============================================================
import json, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0, self.beta_1, self.beta_2, self.beta_U = 50, 8, 3, 2
        self.staff_loading = 5
    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})
    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3, 'demand_U': U})
        params = {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2, 'beta_U': self.beta_U}
        return df, params
    def set_heteroscedasticity(self, s): self.heteroscedasticity_strength = s
    def set_endogeneity(self, s): self.endogeneity_strength = s
    def set_nonlinearity(self, c): self.nonlinearity = bool(c)

class MonteCarloEngine:
    def run(self, estimator_fn, param_name, param_grid, simulator, n_reps=5000, n_obs=500):
        results = np.empty((len(param_grid), n_reps))
        setter = getattr(simulator, f'set_{param_name}', None)
        for i, val in enumerate(param_grid):
            if setter: setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
        return results
    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        return np.array([estimator_fn(dgp_fn(seed=r, n=n_obs)) for r in range(n_reps)])

class DiagnosticSuite:
    @staticmethod
    def run_diagnostics(model_result):
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals'); ax.set_title('Residuals vs Fitted')
        ax = axes[0, 1]
        stats.probplot(resid, dist='norm', plot=ax)
        ax.set_title('Normal Q-Q'); ax.get_lines()[0].set_color(COLORS['ols']); ax.get_lines()[1].set_color(COLORS['bias'])
        ax = axes[1, 0]
        ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values'); ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$'); ax.set_title('Scale-Location')
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage'); ax.set_ylabel('Standardized Residuals'); ax.set_title('Residuals vs Leverage')
        x_range = np.linspace(0.001, ax.get_xlim()[1], 100)
        for cook_val in [0.5, 1.0]:
            for sign in [1, -1]:
                p = model_result.df_model + 1
                y_val = sign * np.sqrt(cook_val * p * (1 - x_range) / x_range)
                ax.plot(x_range, y_val, '--', color=COLORS['bias'], alpha=0.5,
                        label=f"Cook's d={cook_val}" if sign == 1 else None)
        ax.legend(fontsize=8)
        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        results = {}
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const': continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict
        bp_stat, bp_pval, _, _ = het_breuschpagan(model_result.resid, model_result.model.exog)
        results['breusch_pagan'] = (bp_stat, bp_pval)
        results['durbin_watson'] = durbin_watson(model_result.resid)
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)
        return results

class AutopsyReport:
    @staticmethod
    def lightweight(notebook_number):
        threat = widgets.Text(description='Biggest threat:', placeholder='What is the biggest threat to this estimate?',
                              layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        check = widgets.Text(description='How to check:', placeholder='How would you check if that threat is real?',
                             layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, 'threat', threat.value)
                record_trap_response(nb_id, 'check', check.value)
                print('Autopsy responses saved.')
                submit.disabled = True
        submit.on_click(on_submit)
        return widgets.VBox([widgets.HTML(f'<h3>Mini Autopsy \u2014 Notebook {notebook_number}</h3>'), threat, check, submit, output])

    @staticmethod
    def full(notebook_number, available_sidebars=None):
        fields = {
            'point_estimate': widgets.Text(description='Point estimate:', placeholder='My point estimate is:',
                                           layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'robustness': widgets.Text(description='Robustness value:', placeholder='The robustness value is:',
                                       layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'partial_r2': widgets.Text(description='Strongest partial R\u00b2:', placeholder='The strongest observed covariate has partial R\u00b2 of:',
                                       layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'plain_language': widgets.Text(description='Plain language:', placeholder='An omitted variable would need to be ___ times as important as ___ to explain away this result',
                                           layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
        }
        children = [widgets.HTML(f'<h3>Full Autopsy Report \u2014 Notebook {notebook_number}</h3>')]
        children.extend(fields.values())
        if available_sidebars:
            sidebar_dropdown = widgets.Dropdown(options=['(select)'] + list(available_sidebars),
                                                description='Most analogous disaster:', layout=widgets.Layout(width='90%'),
                                                style={'description_width': '180px'})
            children.append(sidebar_dropdown)
            fields['analogous_disaster'] = sidebar_dropdown
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                for key, w in fields.items():
                    record_trap_response(nb_id, key, w.value)
                print('Full autopsy report saved.')
                submit.disabled = True
        submit.on_click(on_submit)
        children.extend([submit, output])
        return widgets.VBox(children)

# ============================================================
# Generate Lalonde-style synthetic data
# ============================================================
def generate_lalonde_data(seed=42):
    """Generate synthetic Lalonde (1986) data with known properties.
    
    Key property: observational OLS estimate of treatment is NEGATIVE (~-$800),
    while true experimental effect is POSITIVE (~+$1,800).
    """
    rng = np.random.default_rng(seed)
    
    TRUE_EFFECT = 1800  # True causal effect of job training
    
    # --- Experimental group (treatment, randomly assigned) ---
    n_treat_exp = 185
    n_control_exp = 260
    
    # Treatment group demographics (disadvantaged workers selected INTO training)
    age_t = rng.normal(25, 6, n_treat_exp).clip(17, 55).astype(int)
    edu_t = rng.normal(10, 1.5, n_treat_exp).clip(3, 16).astype(int)
    black_t = rng.binomial(1, 0.80, n_treat_exp)
    hispanic_t = rng.binomial(1, 0.10, n_treat_exp)
    married_t = rng.binomial(1, 0.15, n_treat_exp)
    nodegree_t = rng.binomial(1, 0.70, n_treat_exp)
    # Very low prior earnings (these are disadvantaged workers)
    re74_t = rng.exponential(2000, n_treat_exp) * rng.binomial(1, 0.55, n_treat_exp)
    re75_t = rng.exponential(1500, n_treat_exp) * rng.binomial(1, 0.60, n_treat_exp)
    
    # Potential outcome without treatment
    base_earn_t = (1500 + 200 * edu_t + 50 * age_t - 800 * nodegree_t
                   + 500 * married_t + 0.15 * re74_t + 0.1 * re75_t
                   + rng.normal(0, 2000, n_treat_exp))
    re78_t = np.maximum(base_earn_t + TRUE_EFFECT, 0)
    
    # Experimental control (SAME population as treatment, randomly assigned)
    age_ce = rng.normal(25, 6, n_control_exp).clip(17, 55).astype(int)
    edu_ce = rng.normal(10, 1.5, n_control_exp).clip(3, 16).astype(int)
    black_ce = rng.binomial(1, 0.80, n_control_exp)
    hispanic_ce = rng.binomial(1, 0.10, n_control_exp)
    married_ce = rng.binomial(1, 0.15, n_control_exp)
    nodegree_ce = rng.binomial(1, 0.70, n_control_exp)
    re74_ce = rng.exponential(2000, n_control_exp) * rng.binomial(1, 0.55, n_control_exp)
    re75_ce = rng.exponential(1500, n_control_exp) * rng.binomial(1, 0.60, n_control_exp)
    base_earn_ce = (1500 + 200 * edu_ce + 50 * age_ce - 800 * nodegree_ce
                    + 500 * married_ce + 0.15 * re74_ce + 0.1 * re75_ce
                    + rng.normal(0, 2000, n_control_exp))
    re78_ce = np.maximum(base_earn_ce, 0)  # No treatment
    
    # --- Observational control group (from CPS/PSID, very different population) ---
    n_control_obs = 260
    age_co = rng.normal(34, 10, n_control_obs).clip(17, 55).astype(int)
    edu_co = rng.normal(12, 2.5, n_control_obs).clip(3, 16).astype(int)
    black_co = rng.binomial(1, 0.25, n_control_obs)
    hispanic_co = rng.binomial(1, 0.08, n_control_obs)
    married_co = rng.binomial(1, 0.70, n_control_obs)
    nodegree_co = rng.binomial(1, 0.25, n_control_obs)
    # Much higher prior earnings (these are regular workers)
    re74_co = rng.normal(14000, 8000, n_control_obs).clip(0, 60000)
    re75_co = rng.normal(13500, 7500, n_control_obs).clip(0, 60000)
    base_earn_co = (3000 + 300 * edu_co + 80 * age_co - 500 * nodegree_co
                    + 800 * married_co + 0.2 * re74_co + 0.15 * re75_co
                    + rng.normal(0, 5000, n_control_obs))
    re78_co = np.maximum(base_earn_co, 0)
    
    # Build experimental dataset
    exp_data = pd.DataFrame({
        'treatment': np.concatenate([np.ones(n_treat_exp), np.zeros(n_control_exp)]),
        'age': np.concatenate([age_t, age_ce]),
        'education': np.concatenate([edu_t, edu_ce]),
        'black': np.concatenate([black_t, black_ce]),
        'hispanic': np.concatenate([hispanic_t, hispanic_ce]),
        'married': np.concatenate([married_t, married_ce]),
        'nodegree': np.concatenate([nodegree_t, nodegree_ce]),
        're74': np.concatenate([re74_t, re74_ce]),
        're75': np.concatenate([re75_t, re75_ce]),
        're78': np.concatenate([re78_t, re78_ce]),
    })
    
    # Build observational dataset (treatment + observational controls)
    obs_data = pd.DataFrame({
        'treatment': np.concatenate([np.ones(n_treat_exp), np.zeros(n_control_obs)]),
        'age': np.concatenate([age_t, age_co]),
        'education': np.concatenate([edu_t, edu_co]),
        'black': np.concatenate([black_t, black_co]),
        'hispanic': np.concatenate([hispanic_t, hispanic_co]),
        'married': np.concatenate([married_t, married_co]),
        'nodegree': np.concatenate([nodegree_t, nodegree_co]),
        're74': np.concatenate([re74_t, re74_co]),
        're75': np.concatenate([re75_t, re75_co]),
        're78': np.concatenate([re78_t, re78_co]),
    })
    
    return obs_data, exp_data, TRUE_EFFECT

obs_data, exp_data, TRUE_EFFECT = generate_lalonde_data(seed=42)

print(f'\u2705 Setup complete.')
print(f'Observational data: {len(obs_data)} observations ({int(obs_data.treatment.sum())} treated, {int((1-obs_data.treatment).sum())} control)')
print(f'Experimental benchmark: {len(exp_data)} observations')
print(f'Variables: {list(obs_data.columns)}')

# Notebook 7: The Real World

**60 minutes** | The capstone

---

## The Lalonde Study

In 1986, Robert LaLonde published one of the most influential papers in economics. He took a **job training program** — the National Supported Work Demonstration — that had been evaluated with a **randomized experiment**, and asked: *what would we have concluded if we only had observational data?*

The answer was devastating for observational research.

You have observational data on individuals who did and did not receive job training. Your job: **estimate the effect of job training on earnings (re78)**.

You have all the standard controls: age, education, race, marital status, prior earnings in 1974 and 1975. You know the tools. You've passed every diagnostic.

Let's see what happens.

In [ ]:
# ============================================================
# Act 1: False Confidence
# ============================================================

# Run OLS: re78 ~ treatment + controls
formula_vars = ['treatment', 'age', 'education', 'black', 'hispanic',
                'married', 'nodegree', 're74', 're75']
X_obs = sm.add_constant(obs_data[formula_vars])
model_obs = sm.OLS(obs_data['re78'], X_obs).fit()

print('='*70)
print('OLS Regression: re78 ~ treatment + age + education + black + hispanic')
print('                       + married + nodegree + re74 + re75')
print('='*70)
print(model_obs.summary())

obs_estimate = model_obs.params['treatment']
print(f'\n\u27a1\ufe0f  Estimated effect of job training: ${obs_estimate:,.0f}')
print(f'   p-value: {model_obs.pvalues["treatment"]:.4f}')

# Run diagnostics
print('\n' + '='*70)
print('Diagnostic Suite')
print('='*70)
fig = DiagnosticSuite.run_diagnostics(model_obs)
plt.show()

tests = DiagnosticSuite.summary_tests(model_obs)
print('\nVIF values:')
for name, vif in tests['vif'].items():
    status = '\u2705' if vif < 3 else '\u26a0\ufe0f'
    print(f'  {status} {name}: {vif:.2f}')

bp_stat, bp_pval = tests['breusch_pagan']
print(f'\nBreusch-Pagan: stat={bp_stat:.2f}, p={bp_pval:.4f}',
      '\u2705 Homoscedastic' if bp_pval > 0.05 else '\u26a0\ufe0f Marginal')

# Try robust SEs
model_robust = sm.OLS(obs_data['re78'], X_obs).fit(cov_type='HC1')
robust_estimate = model_robust.params['treatment']
robust_se = model_robust.bse['treatment']
print(f'\nRobust SEs: treatment = ${robust_estimate:,.0f} (SE = ${robust_se:,.0f})')
print(f'  Change from classical: ${robust_estimate - obs_estimate:,.0f} \u2014 barely moves.')

print('\n' + '='*70)
print('All diagnostics look acceptable. VIFs under 3. No severe issues.')
print('='*70)

# Trap widget
trap = create_trap_widget(
    'notebook_7', 'q1',
    'Based on this analysis, does the job training program work?',
    ['Yes \u2014 training increases earnings',
     'No \u2014 training decreases earnings',
     "Can't tell from this data"]
)
display(trap)

In [ ]:
# ============================================================
# Act 2: The Confrontation
# ============================================================

if not check_gate('notebook_7', 'q1'):
    print('\u26a0\ufe0f Please answer the question in the previous cell before continuing.')
else:
    student_answer = get_trap_response('notebook_7', 'q1')
    print(f'Your answer: "{student_answer}"')
    print()
    
    # Experimental benchmark
    X_exp = sm.add_constant(exp_data[formula_vars])
    model_exp = sm.OLS(exp_data['re78'], X_exp).fit()
    exp_estimate = model_exp.params['treatment']
    
    # Side-by-side comparison
    print('='*70)
    print('THE CONFRONTATION')
    print('='*70)
    print()
    print(f'  Observational estimate:  ${obs_estimate:>10,.0f}')
    print(f'  Experimental estimate:   ${exp_estimate:>10,.0f}')
    print(f'  {"":>25s}  --------')
    print(f'  Discrepancy:             ${exp_estimate - obs_estimate:>10,.0f}')
    print()
    
    sign_obs = 'NEGATIVE' if obs_estimate < 0 else 'POSITIVE'
    sign_exp = 'NEGATIVE' if exp_estimate < 0 else 'POSITIVE'
    print(f'  Observational sign: {sign_obs}')
    print(f'  Experimental sign:  {sign_exp}')
    print()
    print('  Every diagnostic passed. The model looked fine.')
    print(f'  The answer was wrong by over ${abs(exp_estimate - obs_estimate):,.0f} \u2014 in the WRONG DIRECTION.')
    print()
    
    # Bar chart comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: estimates comparison
    ax = axes[0]
    bars = ax.bar(['Observational\n(your estimate)', 'Experimental\n(truth)'],
                  [obs_estimate, exp_estimate],
                  color=[COLORS['bias'], COLORS['repair']],
                  edgecolor='black', linewidth=0.5, width=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Estimated Treatment Effect ($)', fontsize=12)
    ax.set_title('Treatment Effect Estimates', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, [obs_estimate, exp_estimate]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100 * np.sign(bar.get_height()),
                f'${val:,.0f}', ha='center', va='bottom' if val > 0 else 'top', fontsize=13, fontweight='bold')
    
    # Right: density plots of pre-treatment characteristics
    ax = axes[1]
    treat_mask = obs_data['treatment'] == 1
    ctrl_mask = obs_data['treatment'] == 0
    
    # Standardize re74 for visualization
    re74_treat = obs_data.loc[treat_mask, 're74']
    re74_ctrl = obs_data.loc[ctrl_mask, 're74']
    
    bins = np.linspace(0, max(re74_treat.max(), re74_ctrl.max()), 30)
    ax.hist(re74_treat, bins=bins, alpha=0.5, density=True, label='Treatment group',
            color=COLORS['ols'], edgecolor='white')
    ax.hist(re74_ctrl, bins=bins, alpha=0.5, density=True, label='Control group (observational)',
            color=COLORS['bias'], edgecolor='white')
    ax.set_xlabel('Prior Earnings 1974 ($)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Pre-Treatment Earnings: Treatment vs Control', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    
    fig.tight_layout()
    plt.show()
    
    print('\nThe distributions barely overlap.')
    print('The observational control group is fundamentally different from the treated group.')
    print(f'Mean re74 \u2014 Treatment: ${re74_treat.mean():,.0f}  |  Control: ${re74_ctrl.mean():,.0f}')
    print(f'Mean age  \u2014 Treatment: {obs_data.loc[treat_mask, "age"].mean():.1f}    |  Control: {obs_data.loc[ctrl_mask, "age"].mean():.1f}')
    print(f'% Black   \u2014 Treatment: {obs_data.loc[treat_mask, "black"].mean()*100:.0f}%      |  Control: {obs_data.loc[ctrl_mask, "black"].mean()*100:.0f}%')
    print(f'% Married \u2014 Treatment: {obs_data.loc[treat_mask, "married"].mean()*100:.0f}%      |  Control: {obs_data.loc[ctrl_mask, "married"].mean()*100:.0f}%')

In [ ]:
# ============================================================
# Act 2 continued: Interactive subgroup analysis
# ============================================================

# Experimental benchmark for comparison
X_exp = sm.add_constant(exp_data[formula_vars])
model_exp = sm.OLS(exp_data['re78'], X_exp).fit()
exp_estimate = model_exp.params['treatment']

out = widgets.Output()

def update_subgroup(n_bins):
    with out:
        clear_output(wait=True)
        
        # Bin by prior earnings (re74)
        obs_copy = obs_data.copy()
        obs_copy['re74_bin'] = pd.qcut(obs_copy['re74'], q=n_bins, duplicates='drop')
        actual_bins = obs_copy['re74_bin'].nunique()
        
        # Within-bin treatment effects (simple difference in means)
        bin_effects = []
        bin_labels = []
        bin_sizes = []
        for b in sorted(obs_copy['re74_bin'].unique()):
            sub = obs_copy[obs_copy['re74_bin'] == b]
            treat = sub[sub['treatment'] == 1]['re78']
            ctrl = sub[sub['treatment'] == 0]['re78']
            if len(treat) > 2 and len(ctrl) > 2:
                bin_effects.append(treat.mean() - ctrl.mean())
                bin_labels.append(str(b))
                bin_sizes.append(len(sub))
        
        # Weighted average effect
        if bin_effects:
            weights = np.array(bin_sizes) / sum(bin_sizes)
            weighted_effect = np.average(bin_effects, weights=weights)
        else:
            weighted_effect = obs_estimate
        
        gap = exp_estimate - weighted_effect
        
        fig, ax = plt.subplots(figsize=(12, 5))
        x_pos = range(len(bin_effects))
        bars = ax.bar(x_pos, bin_effects, color=COLORS['ols'], alpha=0.7,
                      edgecolor='white', label='Within-bin effect')
        ax.axhline(exp_estimate, color=COLORS['repair'], linewidth=2.5,
                   linestyle='--', label=f'Experimental benchmark: ${exp_estimate:,.0f}',
                   marker='D', markersize=0)
        ax.axhline(weighted_effect, color=COLORS['truth'], linewidth=2,
                   linestyle='-', label=f'Weighted average: ${weighted_effect:,.0f}')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_xlabel('Prior Earnings Bin (re74)', fontsize=12)
        ax.set_ylabel('Treatment Effect ($)', fontsize=12)
        ax.set_title(f'Subgroup Analysis: {actual_bins} bins of prior earnings | '
                     f'Gap remaining: ${gap:,.0f}', fontsize=13, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([f'Bin {i+1}' for i in x_pos], fontsize=9)
        ax.legend(fontsize=11)
        fig.tight_layout()
        plt.show()
        
        print(f'Bins: {actual_bins} | Weighted effect: ${weighted_effect:,.0f} | '
              f'Experimental: ${exp_estimate:,.0f} | Gap remaining: ${gap:,.0f}')
        if abs(gap) > 500:
            print('\n\u27a1\ufe0f Observed covariates help but cannot fully close the gap.')
            print('  Unobserved differences between treated and control groups remain.')

slider = widgets.IntSlider(value=5, min=2, max=20, step=1,
                           description='Number of bins:',
                           style={'description_width': 'initial'},
                           layout=widgets.Layout(width='60%'))

print('Subgroup Analysis: Can observed covariates close the gap?')
print('Adjust the number of bins for prior earnings (re74) and watch the effect estimate.')
print()
display(slider)
display(out)

def on_slider_change(change):
    update_subgroup(change['new'])

slider.observe(on_slider_change, names='value')
update_subgroup(slider.value)

In [ ]:
# ============================================================
# Act 3: Sensitivity Analysis (Cinelli-Hazlett inspired)
# ============================================================

def compute_partial_r2(model_full, var_name, y, X_full_df):
    """Compute partial R-squared of var_name with respect to treatment and outcome."""
    other_vars = [c for c in X_full_df.columns if c != var_name and c != 'const']
    X_reduced = sm.add_constant(X_full_df[other_vars])
    model_reduced = sm.OLS(y, X_reduced).fit()
    
    ss_full = np.sum(model_full.resid**2)
    ss_reduced = np.sum(model_reduced.resid**2)
    partial_r2_outcome = (ss_reduced - ss_full) / ss_reduced
    
    # Partial R2 with treatment
    treat_reduced_vars = [c for c in X_full_df.columns if c != var_name and c != 'const']
    X_treat_red = sm.add_constant(X_full_df[treat_reduced_vars])
    treat_model_full = sm.OLS(X_full_df['treatment'] if 'treatment' in X_full_df.columns
                              else obs_data['treatment'], X_treat_red).fit()
    
    treat_all_vars = [c for c in X_full_df.columns if c != 'const']
    X_treat_full = sm.add_constant(X_full_df[treat_all_vars])
    # For treatment prediction, we include var_name
    treat_model_with = sm.OLS(obs_data['treatment'], sm.add_constant(X_full_df[treat_all_vars])).fit()
    
    ss_t_full = np.sum(treat_model_with.resid**2)
    ss_t_reduced = np.sum(treat_model_full.resid**2)
    partial_r2_treatment = max((ss_t_reduced - ss_t_full) / ss_t_reduced, 0)
    
    return partial_r2_treatment, partial_r2_outcome

# Compute partial R-squared for each covariate
covariates = ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74', 're75']
X_df = obs_data[formula_vars].copy()

partial_r2s = {}
for var in covariates:
    # Partial R2 with outcome (re78)
    other = [v for v in formula_vars if v != var]
    X_full = sm.add_constant(obs_data[formula_vars])
    X_red = sm.add_constant(obs_data[other])
    m_full = sm.OLS(obs_data['re78'], X_full).fit()
    m_red = sm.OLS(obs_data['re78'], X_red).fit()
    pr2_y = (np.sum(m_red.resid**2) - np.sum(m_full.resid**2)) / np.sum(m_red.resid**2)
    
    # Partial R2 with treatment
    other_no_treat = [v for v in formula_vars if v != var and v != 'treatment']
    X_t_full = sm.add_constant(obs_data[[v for v in formula_vars if v != 'treatment']])
    X_t_red = sm.add_constant(obs_data[other_no_treat])
    m_t_full = sm.OLS(obs_data['treatment'], X_t_full).fit()
    m_t_red = sm.OLS(obs_data['treatment'], X_t_red).fit()
    pr2_d = max((np.sum(m_t_red.resid**2) - np.sum(m_t_full.resid**2)) / np.sum(m_t_red.resid**2), 0)
    
    partial_r2s[var] = (pr2_d, pr2_y)

# Compute robustness value
treatment_coef = model_obs.params['treatment']
treatment_se = model_obs.bse['treatment']
treatment_t = treatment_coef / treatment_se
dof = model_obs.df_resid
# RV: approximate as |t|/(|t| + sqrt(dof))
rv = abs(treatment_t) / (abs(treatment_t) + np.sqrt(dof))

print('='*70)
print('SENSITIVITY ANALYSIS (Cinelli-Hazlett Framework)')
print('='*70)
print(f'\nTreatment coefficient: ${treatment_coef:,.0f}')
print(f'Robustness Value (RV): {rv:.3f}')
print(f'  A confounder explaining just {rv*100:.1f}% of the residual variation')
print(f'  in both treatment and outcome would be enough to nullify this result.')
print()
print('Partial R\u00b2 of observed covariates:')
for var, (pr2_d, pr2_y) in sorted(partial_r2s.items(), key=lambda x: max(x[1]), reverse=True):
    print(f'  {var:>12s}: R\u00b2(treatment)={pr2_d:.4f}, R\u00b2(outcome)={pr2_y:.4f}')

# --- Contour plot ---
out2 = widgets.Output()

def update_sensitivity(hyp_r2_treat, hyp_r2_outcome):
    with out2:
        clear_output(wait=True)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Create grid for contours
        r2d_grid = np.linspace(0, 0.5, 200)
        r2y_grid = np.linspace(0, 0.5, 200)
        R2D, R2Y = np.meshgrid(r2d_grid, r2y_grid)
        
        # Bias formula: bias ~ coef * sqrt(R2D * R2Y / ((1-R2D)*(1-R2Y))) * SE_scale
        # Simplified: adjusted_estimate = coef - sign(coef) * |coef| * sqrt(R2D*R2Y) * scale
        se_resid = np.sqrt(np.sum(model_obs.resid**2) / dof)
        n = len(obs_data)
        treat_var = np.var(model_obs.model.exog[:, list(model_obs.model.exog_names).index('treatment')])
        
        # Bias approximation using Cinelli-Hazlett
        bias = np.sign(treatment_coef) * se_resid * np.sqrt(R2D * R2Y / ((1 - R2D) * (1 - R2Y))) / np.sqrt(treat_var * n)
        adjusted = treatment_coef - bias
        
        # Shade region where adjusted estimate crosses zero (bias exceeds estimate)
        ax.contourf(R2D, R2Y, adjusted, levels=[-20000, 0], colors=[COLORS['bias']], alpha=0.15)
        
        # Contour lines for adjusted estimate
        contour_levels = np.linspace(-5000, 5000, 21)
        cs = ax.contour(R2D, R2Y, adjusted, levels=contour_levels, colors='gray', alpha=0.5, linewidths=0.5)
        ax.clabel(cs, inline=True, fontsize=7, fmt='$%.0f')
        
        # Zero contour (boundary)
        ax.contour(R2D, R2Y, adjusted, levels=[0], colors=[COLORS['bias']], linewidths=2)
        
        # Plot observed covariates
        for var, (pr2_d, pr2_y) in partial_r2s.items():
            ax.scatter(pr2_d, pr2_y, s=60, zorder=5, color=COLORS['ols'],
                       edgecolor='black', linewidth=0.5, marker='o')
            ax.annotate(var, (pr2_d, pr2_y), textcoords='offset points',
                        xytext=(5, 5), fontsize=9)
        
        # Hypothetical confounder point
        hyp_bias = np.sign(treatment_coef) * se_resid * np.sqrt(hyp_r2_treat * hyp_r2_outcome / (
            (1 - hyp_r2_treat) * (1 - hyp_r2_outcome))) / np.sqrt(treat_var * n)
        hyp_adjusted = treatment_coef - hyp_bias
        
        ax.scatter(hyp_r2_treat, hyp_r2_outcome, s=150, color=COLORS['truth'],
                   marker='*', zorder=10, edgecolor='black', linewidth=0.5,
                   label=f'Hypothetical confounder')
        
        ax.set_xlabel('Partial R\u00b2 with Treatment', fontsize=12)
        ax.set_ylabel('Partial R\u00b2 with Outcome', fontsize=12)
        ax.set_title(f'Sensitivity Analysis | Bias-adjusted estimate: ${hyp_adjusted:,.0f}',
                     fontsize=13, fontweight='bold')
        
        # RV circle
        theta = np.linspace(0, np.pi/2, 100)
        ax.plot(rv * np.cos(theta), rv * np.sin(theta), '--', color=COLORS['repair'],
                linewidth=1.5, label=f'RV = {rv:.3f}')
        
        ax.legend(fontsize=10, loc='upper right')
        ax.set_xlim(0, 0.5)
        ax.set_ylim(0, 0.5)
        fig.tight_layout()
        plt.show()
        
        print(f'Hypothetical confounder: R\u00b2(treatment)={hyp_r2_treat:.3f}, R\u00b2(outcome)={hyp_r2_outcome:.3f}')
        print(f'Bias-adjusted estimate: ${hyp_adjusted:,.0f}')
        if hyp_adjusted * treatment_coef < 0:
            print('\u26a0\ufe0f The confounder FLIPS the sign of the estimate!')

slider_treat = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01,
                                    description='R\u00b2 with Treatment:',
                                    style={'description_width': 'initial'},
                                    layout=widgets.Layout(width='60%'))
slider_outcome = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01,
                                      description='R\u00b2 with Outcome:',
                                      style={'description_width': 'initial'},
                                      layout=widgets.Layout(width='60%'))

print('\nDrag the sliders to place a hypothetical confounder on the plot.')
print('The bias-adjusted estimate updates live.\n')
display(slider_treat, slider_outcome, out2)

def on_change(change):
    update_sensitivity(slider_treat.value, slider_outcome.value)

slider_treat.observe(on_change, names='value')
slider_outcome.observe(on_change, names='value')
update_sensitivity(slider_treat.value, slider_outcome.value)

In [ ]:
# ============================================================
# Full Autopsy Report
# ============================================================

print('Fill in your autopsy report based on what you learned in this notebook.\n')

autopsy = AutopsyReport.full(
    notebook_number=7,
    available_sidebars=[
        'NB1: Hormone therapy that killed',
        'NB2: Google Flu Trends',
        'NB3: Genome-wide association studies',
        'NB4: Challenger disaster',
        'NB5: Long-Term Capital Management',
        'NB6: Police and crime',
    ]
)
display(autopsy)

In [ ]:
#@title Disaster Sidebar: The Hormone Therapy That Killed (expanded)

# This is the expanded version of the NB1 sidebar, now with full sensitivity analysis.

tab_contents = []

# Tab 1: The Story
story_out = widgets.Output()
with story_out:
    print('THE HORMONE REPLACEMENT THERAPY DISASTER')
    print('='*50)
    print()
    print('For decades, observational studies consistently showed that women')
    print('who took hormone replacement therapy (HRT) after menopause had')
    print('30-50% lower rates of heart disease. Doctors prescribed it widely.')
    print()
    print('Then in 2002, the Women\'s Health Initiative \u2014 a large randomized')
    print('controlled trial \u2014 found the opposite: HRT actually INCREASED')
    print('the risk of heart disease, stroke, and breast cancer.')
    print()
    print('The observational studies were wrong. Not uncertain, not noisy \u2014')
    print('wrong in the wrong direction. The women who chose HRT were')
    print('systematically healthier, wealthier, and more health-conscious.')
    print('The "treatment effect" was really a "healthy user effect."')
    print()
    print('This is exactly what happened with the Lalonde data.')
    print('Selection bias flipped the sign of the estimate.')

# Tab 2: The Math
math_out = widgets.Output()
with math_out:
    print('FORMAL DIAGNOSIS: Omitted Variable Bias + Selection')
    print('='*50)
    print()
    print('The OVB formula applies directly:')
    print()
    print('  E[\u03b2_hat] = \u03b2_true + \u03b2_U \u00b7 \u03b4')
    print()
    print('Where:')
    print('  \u03b2_true = +$1,800 (true effect of HRT / job training)')
    print('  \u03b2_U = effect of "healthy lifestyle" / "employability" on outcome')
    print('  \u03b4 = correlation between treatment and confounder')
    print()
    print('For HRT:')
    print('  \u03b2_HRT = +0.5 (true small harmful effect, relative scale)')
    print('  \u03b2_U = -3.0 (healthy behavior strongly predicts outcomes)')
    print('  \u03c1 = 0.6 (HRT users were much healthier)')
    print('  \u03b4 \u2248 0.6 \u00b7 \u03c3_U/\u03c3_D')
    print()
    print('  E[\u03b2_hat] = 0.5 + (-3.0)(0.6) = 0.5 - 1.8 = -1.3')
    print()
    print('  Sign flipped. Observational study says HRT is protective.')
    print('  Reality: HRT is slightly harmful + healthy user bias.')

# Tab 3: The Simulation
sim_out = widgets.Output()
with sim_out:
    print('SIMULATION: 1,000 Observational Studies')
    print('='*50)
    print()
    
    mc = MonteCarloEngine()
    
    def hrt_dgp(seed, n=500):
        rng = np.random.default_rng(seed)
        # U = healthy lifestyle (unobserved)
        U = rng.standard_normal(n)
        # Treatment correlated with U (healthy women choose HRT)
        treat_prob = 1 / (1 + np.exp(-(0.8 * U + rng.normal(0, 0.5, n))))
        treatment = rng.binomial(1, treat_prob)
        # Outcome: HRT slightly harmful, but U strongly protective
        outcome = 0.5 * treatment - 3.0 * U + rng.normal(0, 1, n)
        return pd.DataFrame({'treatment': treatment, 'outcome': outcome})
    
    def hrt_estimator(df):
        X = sm.add_constant(df[['treatment']])
        return sm.OLS(df['outcome'], X).fit().params['treatment']
    
    run_btn = widgets.Button(description='Run Simulation', button_style='primary')
    sim_plot_out = widgets.Output()
    
    def run_sim(btn):
        with sim_plot_out:
            clear_output(wait=True)
            estimates = mc.quick_run(hrt_estimator, hrt_dgp, n_reps=1000, n_obs=500)
            
            fig, ax = plt.subplots(figsize=(8, 4))
            # 60% saturation colors for sidebar
            ax.hist(estimates, bins=40, color=COLORS['ols'], edgecolor='white', alpha=0.8)
            ax.axvline(0.5, color=COLORS['truth'], linewidth=2.5, linestyle='-',
                       label='True effect (+0.5)', marker='D', markersize=0)
            ax.axvline(np.mean(estimates), color=COLORS['bias'], linewidth=2,
                       linestyle=':', label=f'Mean estimate ({np.mean(estimates):.2f})')
            ax.set_xlabel('Estimated HRT Effect', fontsize=11)
            ax.set_ylabel('Count', fontsize=11)
            ax.set_title('1,000 Observational Studies of HRT', fontsize=12)
            ax.legend(fontsize=10)
            fig.tight_layout()
            plt.show()
            
            pct_wrong_sign = np.mean(estimates < 0) * 100
            print(f'True effect: +0.5 (slightly harmful)')
            print(f'Mean observational estimate: {np.mean(estimates):.2f} (appears protective!)')
            print(f'{pct_wrong_sign:.0f}% of studies got the wrong sign.')
    
    run_btn.on_click(run_sim)
    display(run_btn)
    display(sim_plot_out)

tabs = widgets.Tab(children=[story_out, math_out, sim_out])
tabs.set_title(0, 'The Story')
tabs.set_title(1, 'The Math')
tabs.set_title(2, 'The Simulation')
display(tabs)

---

## Is Regression Hopeless?

No. You just need the right **design**.

Everything you've learned in Notebooks 1–7 has been about what can go wrong with regression when the data is observational — when treatment and control groups aren't comparable, when confounders lurk, when diagnostics pass but the answer is still wrong.

But there are settings where the world gives you something close to a natural experiment. Where the comparison is credible *by design*, not by assumption.

In those settings, regression works brilliantly.

---

*Your coefficient is now correct. But can you trust the design?*

<div style='background: #1a1a2e; color: white; padding: 20px; border-radius: 10px; margin: 20px 0;'>
    <h3 style='color: #D4A017; margin-top: 0;'>Next: Notebook 8 — The Redemption</h3>
    <p>Regression works when the design earns your trust. See how.</p>
    <p><em>"Regression works brilliantly when the world gives you a natural experiment."</em></p>
</div>